# CellIQ CT nodule detector (demo cases)

Runs the open-source MONAI `lung_nodule_ct_detection` bundle (Apache-2.0, trained on LUNA16) on a few LUNA16 CT scans,
checks the boxes against the radiologists' marked nodules, and exports slices + detections for the CellIQ app.

Credit: LUNA16 (https://luna16.grand-challenge.org, CC BY 4.0), built from LIDC-IDRI (CC BY 3.0). Research use only. Not a diagnosis.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai[itk,nibabel,tqdm]", "fire", "pytorch-ignite", "SimpleITK", "pandas"],
                   capture_output=True, text=True)
print("pip exit code:", r.returncode)
import monai, torch, json
print("monai", monai.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
r = subprocess.run([sys.executable, "-m", "monai.bundle", "download", "lung_nodule_ct_detection", "--bundle_dir", "/content/bundles"],
                   capture_output=True, text=True)
print("bundle download exit code:", r.returncode)
BUNDLE = "/content/bundles/lung_nodule_ct_detection"
import os
print(sorted(os.listdir(BUNDLE)), os.path.exists(BUNDLE + "/models/model.pt"))

## Download LUNA16 subset0 and the annotations from Zenodo

In [ ]:
import requests, time
REC_IDS = [3723295, 2595812, 2595813, 2596478, 4121926]
want = {"subset0.zip": None, "annotations.csv": None}
for rid in REC_IDS:
    try:
        rec = requests.get(f"https://zenodo.org/api/records/{rid}", timeout=30).json()
    except Exception as e:
        print(rid, "API error", type(e).__name__); continue
    files = rec.get("files", [])
    print(rid, "->", [f.get("key") for f in files][:14])
    for f in files:
        key = f.get("key") or f.get("filename")
        links = f.get("links") or {}
        if key in want and want[key] is None:
            want[key] = (rid, links.get("self") or links.get("download"), f.get("size"))
print(want)
assert all(want.values()), "Could not find subset0.zip and annotations.csv in the Zenodo records above"

In [ ]:
# Zenodo is slow for the whole 6.8 GB zip, so we only fetch the annotations file and read single scans out of the remote zip.
os.makedirs("/content/luna", exist_ok=True)
rid, link, size = want["annotations.csv"]
rr = subprocess.run(["curl", "-L", "-s", "--retry", "3", "-o", "/content/luna/annotations.csv", link], capture_output=True, text=True)
print("annotations.csv:", rr.returncode, os.path.getsize("/content/luna/annotations.csv"), "bytes")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "remotezip"], capture_output=True, text=True)
print("remotezip install exit code:", r.returncode)

## Pick demo scans (3 with marked nodules, 1 without) and extract only those from the remote zip

In [ ]:
from remotezip import RemoteZip
import pandas as pd
ann = pd.read_csv("/content/luna/annotations.csv")
print("annotations:", len(ann), "nodules in", ann.seriesuid.nunique(), "scans")
z = RemoteZip(want["subset0.zip"][1])
names = z.namelist()
mhd = {os.path.basename(n)[:-4]: n for n in names if n.endswith(".mhd")}
print("scans in subset0:", len(mhd))
inn = ann[ann.seriesuid.isin(mhd)].groupby("seriesuid").agg(n=("diameter_mm", "size"), maxd=("diameter_mm", "max")).sort_values("maxd", ascending=False)
print(inn.head(8))
pick = [inn.index[0], inn.index[len(inn) // 2], inn.index[min(len(inn) - 1, int(len(inn) * 0.8))]]
neg = [u for u in mhd if u not in set(ann.seriesuid)][:1]
cases = pick + neg
print("chosen:", cases)
os.makedirs("/content/cases", exist_ok=True)
datalist = []
for u in cases:
    for ext in (".mhd", ".raw"):
        src = mhd[u][:-4] + ext
        t0 = time.time()
        z.extract(src, "/content/cases")
        print("extracted", src, round(os.path.getsize("/content/cases/" + src) / 1e6), "MB", round(time.time() - t0), "s", flush=True)
    datalist.append({"image": mhd[u]})
json.dump({"validation": datalist}, open("/content/datalist.json", "w"))
print(datalist)

## Run the detector (raw LUNA16 mode: the bundle resamples to 0.703 x 0.703 x 1.25 mm itself)

In [ ]:
cfg = json.load(open(BUNDLE + "/configs/inference.json"))
cfg["whether_raw_luna16"] = True
cfg["dataset_dir"] = "/content/cases"
cfg["data_list_file_path"] = "/content/datalist.json"
cfg["output_dir"] = "/content/det_out"
cfg["output_filename"] = "result.json"
cfg["dataloader"]["num_workers"] = 0          # 4 workers ran out of RAM on Colab
os.makedirs("/content/det_out", exist_ok=True)   # the bundle does not create its output folder
json.dump(cfg, open("/content/infer_cfg.json", "w"), indent=1)
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "monai.bundle", "run", "--config_file", "/content/infer_cfg.json",
                    "--meta_file", BUNDLE + "/configs/metadata.json"], cwd=BUNDLE, capture_output=True, text=True)
print("inference exit code:", r.returncode, f"| {time.time()-t0:.0f}s")
print((r.stdout + r.stderr)[-2500:])

In [ ]:
import glob
outs = glob.glob("/content/det_out/**/*.json", recursive=True)
print("result files:", outs)
res = json.load(open(outs[0]))
print(type(res).__name__, (list(res.keys())[:10] if isinstance(res, dict) else len(res)))
print(json.dumps(res, default=str)[:1500])

## Compare with the radiologists' nodules, and export slices + detections for the app

In [ ]:
import numpy as np, SimpleITK as sitk
from PIL import Image, ImageDraw

def unwrap(d):
    if isinstance(d, dict):
        for k in ("validation", "results", "data", "predictions"):
            if k in d and isinstance(d[k], list):
                return d[k]
        return [d]
    return d

def pick(rec, keys):
    for k in keys:
        if isinstance(rec, dict) and k in rec:
            return rec[k]
    return None

recs = unwrap(res)
print("records:", len(recs))
detections = {}
for idx, rec in enumerate(recs):
    img_ref = str(pick(rec, ["image", "image_path", "filename", "path"]) or "")
    uid = next((u for u in cases if u in img_ref), cases[idx] if idx < len(cases) else None)
    boxes = np.array(pick(rec, ["box", "boxes", "pred_box"]) or [], dtype=float).reshape(-1, 6)
    scores = np.array(pick(rec, ["label_scores", "scores", "score", "label_score"]) or [], dtype=float).reshape(-1)
    detections[uid] = (boxes, scores)
    print(uid, "boxes:", len(boxes), "top scores:", np.round(np.sort(scores)[::-1][:5], 3))

In [ ]:
SIZE = 256
os.makedirs("/content/export", exist_ok=True)
summary = {"model": "MONAI lung_nodule_ct_detection v0.6.9 (Apache-2.0)", "data": "LUNA16 subset0 (CC BY 4.0) / LIDC-IDRI (CC BY 3.0)",
           "cases": [], "thresholds": {}}
THRS = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
stats = {t: {"found": 0, "total": 0, "unmatched": 0} for t in THRS}

for ci, uid in enumerate(cases, start=1):
    itk = sitk.ReadImage(f"/content/cases/{mhd[uid]}")
    vol = sitk.GetArrayFromImage(itk).astype(np.float32)          # z, y, x in HU
    sp = itk.GetSpacing()                                          # x, y, z (mm)
    boxes, scores = detections[uid]
    # detector boxes are centre x,y,z + width,height,depth in world mm and ALREADY in the LUNA16 (LPS) coordinate system: no flip
    centres_lps = boxes[:, :3] if len(boxes) else np.zeros((0, 3))
    a = ann[ann.seriesuid == uid]
    a_xyz = a[["coordX", "coordY", "coordZ"]].values if len(a) else np.zeros((0, 3))
    a_rad = (a["diameter_mm"].values / 2.0) if len(a) else np.zeros(0)
    for t in THRS:
        keep = scores >= t
        hit = np.zeros(len(a_xyz), bool); used = np.zeros(int(keep.sum()), bool)
        for k, c in enumerate(centres_lps[keep]):
            d = np.linalg.norm(a_xyz - c, axis=1) if len(a_xyz) else np.zeros(0)
            m = d <= a_rad
            if m.any():
                hit |= m; used[k] = True
        stats[t]["found"] += int(hit.sum()); stats[t]["total"] += len(a_xyz); stats[t]["unmatched"] += int((~used).sum())

    # export: lung-window slices (HU -1000..300), downsampled, plus detections in slice-pixel coordinates
    os.makedirs(f"/content/export/case_{ci}", exist_ok=True)
    win = np.clip((vol + 1000.0) / 1300.0, 0, 1)
    for k in range(win.shape[0]):
        im = Image.fromarray((win[k] * 255).astype("uint8")).resize((SIZE, SIZE), Image.BILINEAR)
        im.save(f"/content/export/case_{ci}/{k:03d}.jpg", quality=80)
    dets = []
    for b, c, s in sorted(zip(boxes, centres_lps, scores), key=lambda x: -x[2]):
        if s < 0.1:
            continue
        ix, iy, iz = itk.TransformPhysicalPointToContinuousIndex([float(c[0]), float(c[1]), float(c[2])])
        dets.append({"score": round(float(s), 3), "center_px": [round(ix * SIZE / vol.shape[2], 1), round(iy * SIZE / vol.shape[1], 1)],
                     "size_px": [round(b[3] / sp[0] * SIZE / vol.shape[2], 1), round(b[4] / sp[1] * SIZE / vol.shape[1], 1)],
                     "slice": int(round(iz)), "depth_slices": round(float(b[5] / sp[2]), 1), "diameter_mm": round(float(max(b[3], b[4])), 1)})
    anns = []
    for xyz, r_ in zip(a_xyz, a_rad):
        ix, iy, iz = itk.TransformPhysicalPointToContinuousIndex([float(xyz[0]), float(xyz[1]), float(xyz[2])])
        anns.append({"center_px": [round(ix * SIZE / vol.shape[2], 1), round(iy * SIZE / vol.shape[1], 1)], "slice": int(round(iz)),
                     "diameter_mm": round(float(r_ * 2), 1)})
    meta = {"case": f"case_{ci}", "seriesuid": uid, "n_slices": int(vol.shape[0]), "slice_size": SIZE, "spacing_mm": [round(s_, 3) for s_ in sp],
            "detections": dets, "radiologist_nodules": anns}
    json.dump(meta, open(f"/content/export/case_{ci}/meta.json", "w"), indent=1)
    summary["cases"].append({"case": f"case_{ci}", "slices": int(vol.shape[0]), "marked_nodules": len(anns), "detections_ge_0.1": len(dets),
                             "top_score": dets[0]["score"] if dets else None})
    print(f"case_{ci}: {vol.shape[0]} slices, {len(anns)} marked nodules, {len(dets)} detections >= 0.1")

for t in THRS:
    s = stats[t]
    summary["thresholds"][str(t)] = {"nodules_found": s["found"], "marked_nodules": s["total"],
                                     "sensitivity": round(s["found"] / max(s["total"], 1), 3),
                                     "unmatched_detections_per_scan": round(s["unmatched"] / len(cases), 2)}
    print(f"score >= {t}: found {s['found']}/{s['total']} marked nodules | unmatched detections per scan {s['unmatched']/len(cases):.1f}")
summary["note"] = ("Only 4 scans from LUNA16 subset0, which I believe is the released model's validation fold (not verified). "
                   "Unmatched detections include real small nodules that the LUNA16 reference does not list.")
json.dump(summary, open("/content/export/summary.json", "w"), indent=1)

In [ ]:
import shutil
shutil.make_archive("/content/cellIQ_ct_demo", "zip", "/content/export")
print("Zip ready: /content/cellIQ_ct_demo.zip", round(os.path.getsize("/content/cellIQ_ct_demo.zip") / 1e6, 1), "MB")
print(json.dumps(summary, indent=1)[:2500])